# LIMIAR — Notebook 8: Integração Experimental (WESAD)

**Objetivo:** transformar o modelo `Random_Forest` experimental gerado no Notebook 7 em um módulo de inferência de características e janelas de EDA/ACC. Este notebook implementa um **protótipo técnico**, não um sistema de alerta clínico.

**Atenção:** o WESAD contém classes experimentais `baseline` e `stress` e **não** dados rotulados de sobrecarga sensorial, meltdown ou shutdown em pessoas autistas.

Nesta etapa não treinamos novos modelos, nem estimamos acurácia sobre o dataset já usado no treino. A avaliação científica de generalização permanece no Notebook 7 (LOPO por participante).

## 1 — Preparar ambiente

O notebook roda no Google Colab. Para abrir dados do Drive, autorize a montagem. As dependências são instaladas apenas se faltarem.

In [ ]:
%pip -q install "scikit-learn>=1.5,<1.9" "fastapi>=0.115,<1" "uvicorn>=0.30,<1" "joblib>=1.4,<2" "pytest>=8,<10"

In [ ]:
from pathlib import Path
import sys, os, json, time, shutil, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    from google.colab import drive, files
    drive.mount('/content/drive')
except ImportError:
    print('Executando fora do Colab; usaremos arquivos locais.')

ROOT = Path('/content/drive/MyDrive/Projeto Integrador VI/LIMIAR')
if not ROOT.exists():
    candidatos = list(Path('/content/drive/MyDrive').rglob('features_consolidado.csv')) if Path('/content/drive/MyDrive').exists() else []
    if candidatos:
        ROOT = candidatos[0].parent.parent
    else:
        ROOT = Path.cwd()

ARQUIVO_CSV = ROOT / 'Processados' / 'features_consolidado.csv'
ARQUIVO_MODELO = ROOT / 'Modelos' / 'LIMIAR_N7_Random_Forest_experimental.joblib'
if not ARQUIVO_CSV.exists() or not ARQUIVO_MODELO.exists():
    print('Arquivos não encontrados na pasta esperada. Verifique a pasta LIMIAR ou faça upload:')
    print('CSV:', ARQUIVO_CSV, ARQUIVO_CSV.exists())
    print('Modelo:', ARQUIVO_MODELO, ARQUIVO_MODELO.exists())
    try:
        uploaded = files.upload()
        if 'features_consolidado.csv' in uploaded:
            ARQUIVO_CSV = Path('features_consolidado.csv')
        if 'LIMIAR_N7_Random_Forest_experimental.joblib' in uploaded:
            ARQUIVO_MODELO = Path('LIMIAR_N7_Random_Forest_experimental.joblib')
    except NameError:
        pass
assert ARQUIVO_CSV.exists(), 'CSV consolidado não encontrado.'
assert ARQUIVO_MODELO.exists(), 'Modelo RF N7 não encontrado.'

PASTA_RESULTADOS = (ROOT / 'Integracao_Experimental') if (ROOT / 'Processados').exists() else Path('LIMIAR_N8_Resultados')
PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
print('Projeto:', ROOT)
print('Modelo:', ARQUIVO_MODELO)
print('Dados:', ARQUIVO_CSV)
print('Saída:', PASTA_RESULTADOS)

## 2 — Criar o módulo de inferência

O código seguinte é idêntico ao `limiar_inferencia.py` entregue junto do pacote. Ele valida ordem das features e entradas numéricas, não lê rótulos e produz scores experimentais.

**Segurança:** arquivos `.joblib` devem ser carregados somente quando produzidos por fonte confiável, pois a desserialização pode executar código.

In [ ]:
%%writefile limiar_inferencia.py
"""LIMIAR: inferência EXPERIMENTAL (baseline/stress do WESAD).

Não é um detector validado de sobrecarga sensorial, meltdown ou shutdown.
Carregar joblib somente de fontes confiáveis: desserialização executa código.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Any, Mapping, Sequence

import joblib
import numpy as np
import pandas as pd

FS_EDA = 4
FS_ACC = 32
JANELA_S = 30
PASSO_S = 15
AMOSTRAS_EDA = JANELA_S * FS_EDA
AMOSTRAS_ACC = JANELA_S * FS_ACC
FEATURES_PERMITIDAS = {
    'BVP_mean', 'BVP_std', 'BVP_min', 'BVP_max',
    'EDA_mean', 'EDA_std', 'EDA_min', 'EDA_max',
    'ACC_mean', 'ACC_std', 'ACC_max',
}
FEATURES_SENSOR_2 = {
    'EDA_mean', 'EDA_std', 'EDA_min', 'EDA_max',
    'ACC_mean', 'ACC_std', 'ACC_max',
}
AVISO = (
    'Protótipo experimental de identificação de padrões associados a baseline/stress '
    'do WESAD. NÃO validado para sobrecarga sensorial, nem adequado para alertas clínicos.'
)


def carregar_bundle(caminho: str | Path) -> dict[str, Any]:
    """Somente para modelo já treinado e confiável produzido pela equipe."""
    bundle = joblib.load(caminho)
    if not isinstance(bundle, dict):
        raise ValueError('Formato não suportado: esperado bundle com metadados.')
    campos = {'modelo', 'algoritmo', 'features', 'limiar', 'personalizado'}
    if not campos.issubset(bundle):
        raise ValueError(f'Bundle incompleto: {sorted(campos-set(bundle))}')
    features = bundle['features']
    if not isinstance(features, (list, tuple)) or not features or len(features) != len(set(features)):
        raise ValueError('Lista de features ausente, vazia ou duplicada.')
    if not set(features).issubset(FEATURES_PERMITIDAS):
        raise ValueError('Bundle solicita features não reconhecidas.')
    if bundle['personalizado']:
        raise ValueError('Este serviço não suporta modelo personalizado sem protocolo de calibração.')
    modelo = bundle['modelo']
    if not hasattr(modelo, 'predict'):
        raise ValueError('O modelo salvo não possui predict().')
    n = getattr(modelo, 'n_features_in_', None)
    if n is not None and int(n) != len(features):
        raise ValueError('Quantidade de features incompatível com o modelo salvo.')
    if not np.isfinite(float(bundle['limiar'])):
        raise ValueError('Limiar inválido.')
    return bundle


def _validar_frame(frame: pd.DataFrame, features: Sequence[str]) -> pd.DataFrame:
    ausentes = [c for c in features if c not in frame.columns]
    if ausentes:
        raise ValueError(f'Features ausentes: {ausentes}')
    if frame.empty:
        raise ValueError('O dataset está vazio.')
    dados = frame.loc[:, list(features)].apply(pd.to_numeric, errors='raise')
    valores = dados.to_numpy(dtype=float)
    if not np.isfinite(valores).all():
        raise ValueError('Features contêm NaN, infinito ou valores inválidos.')
    return dados.astype(float)


def pontuar(bundle: dict[str, Any], frame: pd.DataFrame) -> np.ndarray:
    features = bundle['features']
    dados = _validar_frame(frame, features)
    nome = bundle['algoritmo']
    if nome.startswith('IF_'):
        return -np.asarray(bundle['modelo'].decision_function(dados.to_numpy()), dtype=float)
    if nome in {'Random_Forest', 'Regressao_Logistica'}:
        return np.asarray(bundle['modelo'].predict_proba(dados.to_numpy())[:, 1], dtype=float)
    raise ValueError(f'Algoritmo não suportado: {nome}')


def inferir_features(bundle: dict[str, Any], frame: pd.DataFrame) -> pd.DataFrame:
    """Aceita features previamente calculadas; ignora completamente qualquer label presente."""
    s = pontuar(bundle, frame)
    limiar = float(bundle['limiar'])
    y = (s > limiar).astype(int)
    resultado = pd.DataFrame({
        'score_modelo': s,
        'limiar': limiar,
        'indicador_experimental': y,
        'interpretacao': np.where(
            y == 1, 'padrao_stress_experimental_WESAD', 'padrao_baseline_experimental_WESAD'
        ),
    }, index=frame.index)
    for coluna in ['participante', 'janela', 'inicio_s', 'fim_s']:
        if coluna in frame.columns:
            resultado.insert(len(resultado.columns), coluna, frame[coluna])
    resultado['uso'] = 'APENAS_PESQUISA'
    return resultado


def extrair_features_janela(eda: Sequence[float], acc: Sequence[Sequence[float]], *,
                             unidades_eda: str = 'WESAD_ORIGINAL',
                             unidades_acc: str = 'WESAD_ORIGINAL') -> dict[str, float]:
    """30 s; EDA 4 Hz (120) e ACC tri-axial 32 Hz (960x3).

    Requer sinais nas mesmas unidades/escala do WESAD original. Não faz conversão
    automática nem resample; usar outra pulseira exige validar essa compatibilidade.
    """
    if unidades_eda != 'WESAD_ORIGINAL' or unidades_acc != 'WESAD_ORIGINAL':
        raise ValueError('Unidades incompatíveis: valide/converta os sensores antes da inferência.')
    e = np.asarray(eda, dtype=float)
    a = np.asarray(acc, dtype=float)
    if e.shape != (AMOSTRAS_EDA,) or a.shape != (AMOSTRAS_ACC, 3):
        raise ValueError(
            f'Esperado EDA ({AMOSTRAS_EDA},) e ACC ({AMOSTRAS_ACC},3); '
            f'recebido EDA {e.shape}, ACC {a.shape}'
        )
    if not np.isfinite(e).all() or not np.isfinite(a).all():
        raise ValueError('Sinais inválidos: NaN/infinito.')
    magnitude = np.sqrt(np.sum(a*a, axis=1))
    return {
        'EDA_mean': float(e.mean()),
        'EDA_std': float(e.std(ddof=0)),
        'EDA_min': float(e.min()),
        'EDA_max': float(e.max()),
        'ACC_mean': float(magnitude.mean()),
        'ACC_std': float(magnitude.std(ddof=0)),
        'ACC_max': float(magnitude.max()),
    }


def inferir_sensores(bundle: dict[str, Any], eda: Sequence[float],
                     acc: Sequence[Sequence[float]], *,
                     unidades_eda: str = 'WESAD_ORIGINAL',
                     unidades_acc: str = 'WESAD_ORIGINAL') -> dict[str, Any]:
    if not set(bundle['features']).issubset(FEATURES_SENSOR_2):
        raise ValueError(
            'O modelo necessita BVP ou de outras features; esta interface usa somente EDA+ACC.'
        )
    f = extrair_features_janela(
        eda, acc, unidades_eda=unidades_eda, unidades_acc=unidades_acc
    )
    r = inferir_features(bundle, pd.DataFrame([f])).iloc[0]
    return {
        'score_modelo': float(r['score_modelo']),
        'limiar': float(r['limiar']),
        'indicador_experimental': int(r['indicador_experimental']),
        'interpretacao': str(r['interpretacao']),
        'features_utilizadas': {k: float(f[k]) for k in bundle['features']},
        'aviso': AVISO,
    }


def marcar_sequencias(resultado: pd.DataFrame, consecutivas: int = 3) -> pd.DataFrame:
    """Marca persistência no replay: somente janelas em ordem e com passo de 15 s.

    Isto não é calibrado/validado para uso clínico; é um recurso de visualização.
    """
    if consecutivas < 1:
        raise ValueError('consecutivas precisa ser >= 1')
    if not {'participante', 'inicio_s', 'indicador_experimental'}.issubset(resultado.columns):
        raise ValueError('O replay precisa de participante, inicio_s e indicador_experimental.')
    d = resultado.copy().sort_values(['participante', 'inicio_s'])
    streak = []
    for _, g in d.groupby('participante', sort=False):
        c = 0
        anterior = None
        for _, r in g.iterrows():
            tempo = float(r['inicio_s'])
            contiguo = anterior is not None and abs(tempo-anterior-PASSO_S) < 1e-6
            if int(r['indicador_experimental']) == 1:
                c = (c + 1) if contiguo else 1
            else:
                c = 0
            streak.append(c)
            anterior = tempo
    d['janelas_consecutivas'] = streak
    d['sequencia_experimental'] = (d['janelas_consecutivas'] >= consecutivas).astype(int)
    return d.sort_index()


def main() -> None:
    parser = argparse.ArgumentParser(description='Replay experimental LIMIAR a partir de features CSV')
    parser.add_argument('--modelo', required=True)
    parser.add_argument('--csv', required=True)
    parser.add_argument('--saida', required=True)
    parser.add_argument('--consecutivas', type=int, default=3)
    args = parser.parse_args()
    bundle = carregar_bundle(args.modelo)
    data = pd.read_csv(args.csv)
    predito = inferir_features(bundle, data)
    if {'participante', 'inicio_s'}.issubset(predito.columns):
        predito = marcar_sequencias(predito, args.consecutivas)
    Path(args.saida).parent.mkdir(parents=True, exist_ok=True)
    predito.to_csv(args.saida, index=False)
    print(json.dumps({'janelas': len(predito), 'arquivo': args.saida,
                      'modelo': bundle['algoritmo'], 'features': bundle['features'],
                      'aviso': AVISO}, ensure_ascii=False, indent=2))


if __name__ == '__main__':
    main()

In [ ]:
from limiar_inferencia import (carregar_bundle, inferir_features, inferir_sensores,
                                extrair_features_janela, marcar_sequencias, AVISO)
bundle = carregar_bundle(ARQUIVO_MODELO)
print('Algoritmo:', bundle['algoritmo'])
print('Features esperadas:', bundle['features'])
print('Limiar:', bundle['limiar'])
print(AVISO)

## 3 — Replay técnico do CSV real

Aplicamos a inferência às janelas baseline/stress do WESAD. **Não** calculamos acurácia, F1 ou recall com esse replay: o modelo final foi treinado usando esses mesmos dados. Para avaliação de generalização, use as métricas LOPO do Notebook 7.

In [ ]:
dados = pd.read_csv(ARQUIVO_CSV)
replay = dados[dados['label'].isin([1, 2])].drop(columns=['label']).copy().reset_index(drop=True)
previsoes = marcar_sequencias(inferir_features(bundle, replay))
print('Janelas processadas:', len(previsoes))
display(previsoes.head())
previsoes.to_csv(PASTA_RESULTADOS / 'LIMIAR_N8_replay_inferencia_experimental.csv', index=False)

## 4 — Extração de características de uma janela de sensores

Este exemplo usa sinais **sintéticos** apenas para testar o formato. Para sinais reais, a EDA precisa conter 120 medições a 4 Hz, o ACC precisa conter 960 trios a 32 Hz e os valores devem estar na **escala e unidades compatíveis com os dados originais do WESAD**. O código não sincroniza sensores nem converte unidades.

In [ ]:
rng = np.random.default_rng(42)
eda_sintetica = rng.normal(1.2, .1, size=120)
acc_sintetico = rng.normal(loc=[0., 0., 64.], scale=[3., 3., 3.], size=(960, 3))
features_exemplo = extrair_features_janela(eda_sintetica, acc_sintetico)
display(pd.DataFrame([features_exemplo]))
resultado_exemplo = inferir_sensores(bundle, eda_sintetica, acc_sintetico)
print(json.dumps(resultado_exemplo, indent=2, ensure_ascii=False))

## 5 — Gráfico da execução experimental

A visualização abaixo utiliza janelas extraídas do mesmo WESAD empregado no treino; é ilustrativa do funcionamento da API de inferência, **não** uma medida de desempenho em dados desconhecidos.

In [ ]:
ex = previsoes[previsoes['participante']=='S2'].sort_values('inicio_s')
fig, ax=plt.subplots(figsize=(11,4))
ax.plot(ex['inicio_s']/60, ex['score_modelo'], linewidth=1)
ax.axhline(float(bundle['limiar']), linestyle='--', label='Limiar exploratório')
ax.set(xlabel='Minutos desde o início', ylabel='Score do modelo',
       title='Replay experimental (S2, dados já usados no treino)')
ax.grid(alpha=.2)
ax.legend()
fig.tight_layout()
fig.savefig(PASTA_RESULTADOS / 'LIMIAR_N8_replay_score_S2.png', dpi=170)
plt.show()

## 6 — Medir latência de execução

Medimos apenas o custo computacional do pipeline na CPU da sessão. **O tempo em Colab não prediz desempenho em celular** e não inclui transmissão Bluetooth, coleta dos 30 segundos, sincronização ou execução no backend real.

In [ ]:
linha = replay.iloc[[0]]
for _ in range(5): inferir_features(bundle, linha)
latencias = []
for _ in range(40):
    inicio=time.perf_counter_ns()
    inferir_features(bundle, linha)
    latencias.append((time.perf_counter_ns()-inicio)/1e6)
print('Mediana (ms):', round(float(np.median(latencias)), 3))
print('P95 (ms):', round(float(np.percentile(latencias, 95)), 3))
print('P99 (ms):', round(float(np.percentile(latencias, 99)), 3))
print('Observação: tempo de execução ≠ 30 s para coletar a janela.')

## 7 — API HTTP local (FastAPI)

Escrevemos o arquivo `api_limiar.py` com as rotas `/health`, `/v1/predicao/features` e `/v1/predicao/sensores`. Não há autenticação, controle de tráfego nem consentimento implementados: **não exponha esse serviço na internet**.

In [ ]:
%%writefile api_limiar.py
"""API experimental do LIMIAR: executar localmente com uvicorn api_limiar:app.

Necessário apontar LIMIAR_MODEL_PATH para .joblib confiável da equipe.
Nenhum endpoint representa diagnóstico, recomendação de saúde ou alerta clínico.
"""
import os
from functools import lru_cache
from typing import Literal

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

from limiar_inferencia import AVISO, carregar_bundle, inferir_features, inferir_sensores
import pandas as pd

app = FastAPI(title='LIMIAR — API experimental WESAD', version='0.1.0')

@lru_cache(maxsize=1)
def get_modelo():
    path = os.getenv('LIMIAR_MODEL_PATH')
    if not path:
        raise RuntimeError('Defina LIMIAR_MODEL_PATH com um modelo confiável da equipe.')
    return carregar_bundle(path)

class FeatureRequest(BaseModel):
    caracteristicas: dict[str, float]

class SensorRequest(BaseModel):
    eda: list[float] = Field(min_length=120, max_length=120)
    acc: list[list[float]] = Field(min_length=960, max_length=960)
    unidades_eda: Literal['WESAD_ORIGINAL']
    unidades_acc: Literal['WESAD_ORIGINAL']

@app.get('/health')
def health():
    try:
        b = get_modelo()
        return {'status':'ok', 'algoritmo':b['algoritmo'], 'features':b['features'], 'aviso':AVISO}
    except Exception as exc:
        raise HTTPException(status_code=503, detail=str(exc))

@app.post('/v1/predicao/features')
def predicao_features(body: FeatureRequest):
    try:
        row = inferir_features(get_modelo(), pd.DataFrame([body.caracteristicas])).iloc[0]
        return {'score_modelo':float(row['score_modelo']),
                'limiar':float(row['limiar']),
                'indicador_experimental':int(row['indicador_experimental']),
                'interpretacao':str(row['interpretacao']), 'aviso':AVISO}
    except (ValueError, TypeError) as exc:
        raise HTTPException(status_code=422, detail=str(exc))

@app.post('/v1/predicao/sensores')
def predicao_sensores(body: SensorRequest):
    try:
        return inferir_sensores(get_modelo(), body.eda, body.acc,
                                unidades_eda=body.unidades_eda,
                                unidades_acc=body.unidades_acc)
    except (ValueError, TypeError) as exc:
        raise HTTPException(status_code=422, detail=str(exc))

In [ ]:
from api_limiar import app, get_modelo
from fastapi.testclient import TestClient
os.environ['LIMIAR_MODEL_PATH'] = str(ARQUIVO_MODELO)
get_modelo.cache_clear()
cliente = TestClient(app)
resposta = cliente.get('/health')
print('GET /health:', resposta.status_code, resposta.json())
exemplo = {'caracteristicas': {f: float(replay.iloc[0][f]) for f in bundle['features']}}
resposta = cliente.post('/v1/predicao/features', json=exemplo)
print('POST /v1/predicao/features:', resposta.status_code, resposta.json())
assert resposta.status_code == 200

## 8 — Salvar artefatos e gerar ZIP

Os resultados ficam na pasta `Integracao_Experimental` do Drive (quando ela estiver acessível) ou na pasta local de resultados. Copiamos também a biblioteca e a API, para possibilitar integração posterior ao backend.

In [ ]:
for arq in ['limiar_inferencia.py', 'api_limiar.py']:
    shutil.copy2(arq, PASTA_RESULTADOS / arq)
manifesto = {
    'nome': 'LIMIAR_N8_prototipo_inferencia',
    'modelo': bundle['algoritmo'],
    'features': bundle['features'],
    'limiar': float(bundle['limiar']),
    'janela_s': 30,
    'passo_s': 15,
    'eda_hz': 4,
    'acc_hz': 32,
    'replay_janelas': len(previsoes),
    'mediana_latencia_local_ms': float(np.median(latencias)),
    'p95_latencia_local_ms': float(np.percentile(latencias, 95)),
    'validacao_clinica': False,
    'alerta': AVISO,
}
(PASTA_RESULTADOS/'LIMIAR_N8_manifesto_integracao.json').write_text(
    json.dumps(manifesto, ensure_ascii=False, indent=2), encoding='utf-8'
)
arquivo_zip = Path.cwd() / 'LIMIAR_N8_Artefatos.zip'
with zipfile.ZipFile(arquivo_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(PASTA_RESULTADOS.iterdir()):
        if p.is_file():
            z.write(p, arcname=p.name)
print('Arquivos gerados:', [p.name for p in PASTA_RESULTADOS.iterdir()])
print('Pacote ZIP:', arquivo_zip)
try:
    files.download(str(arquivo_zip))
except Exception:
    pass

## 9 — Próxima integração com o aplicativo real

Este protótipo **não** conecta a pulseiras ou ao frontend existente. Para adaptar ao aplicativo precisamos documentar dispositivo (fabricante e SDK), formato e unidades dos sinais, transporte para o backend, buffer e sincronização das janelas, autenticação, segurança e LGPD. Testes em usuário final requerem avaliação ética apropriada; o WESAD não valida a detecção de sobrecarga sensorial autista.